In [4]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=data['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=data['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
Np_str='1e6'
#Restricts the timesteps of the data from timesteps0 to 140
res='1km'
job_array=False;index_adjust=0
ocean_fraction=0.25

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [5]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

Top 10 objects with highest memory usage
{'initiate_array_2D': '0.0 MB', 'initiate_array_4D': '0.0 MB', 'Ddt': '0.0 MB', 'Ddz': '0.0 MB', 'Ddz_4DStretch': '0.0 MB', 'Ddy_4D': '0.0 MB', 'Ddx_4d': '0.0 MB', 'Ddz_3D': '0.0 MB', 'Ddy_3D': '0.0 MB', 'Ddx_3D': '0.0 MB'}

0.0 GB in use overall


In [3]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [4]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=2
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 16667, end_job = 33334


In [5]:
#Indexing Array with JobArray
parcel=parcel.isel(xh=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [6]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][:,start_job:end_job] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][:,start_job:end_job].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [ ]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min.h5'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['A_g', 'A_c', 'W', 'QCQI', 'Z', 'Y', 'X', 'z']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, W, QCQI, Z, Y, X, parcel_z = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [ ]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
# in_file=dir2+f'VARS_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'VARS_binary_array_{res}_{Np_str}_1min.h5'
in_file=dir2+f'VARS_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['QV','TH']
data_dict = make_data_dict(var_names,read_type)
QV, TH = (data_dict[k] for k in var_names)

In [11]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
# in_file=dir2+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_1min.h5'
in_file=dir2+f'W_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['WB_HADV','WB_VADV','WB_HIDIFF','WB_VIDIFF','WB_HTURB','WB_VTURB','WB_PGRAD','WB_BUOY']
data_dict = make_data_dict(var_names,read_type)
WB_HADV, WB_VADV, WB_HIDIFF, WB_VIDIFF, WB_HTURB, WB_VTURB, WB_PGRAD, WB_BUOY = (data_dict[k] for k in var_names)

In [ ]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
# in_file=dir2+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_1min.h5'
in_file=dir2+f'QV_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['QVB_HADV','QVB_VADV','QVB_HIDIFF','QVB_VIDIFF','QVB_HTURB','QVB_VTURB','QVB_MP']
data_dict = make_data_dict(var_names,read_type)
QVB_HADV, QVB_VADV, QVB_HIDIFF, QVB_VIDIFF, QVB_HTURB, QVB_VTURB, QVB_MP = (data_dict[k] for k in var_names)

In [ ]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
# in_file=dir2+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_5min.h5'
# in_file=dir2+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_1min.h5'
in_file=dir2+f'TH_BUDGET_VARS_binary_array_{res}_{Np_str}_1min_50M.h5'

var_names = ['PTB_HADV','PTB_VADV','PTB_HIDIFF','PTB_VIDIFF','PTB_HTURB','PTB_VTURB','PTB_MP','PTB_RAD','PTB_DIV','PTB_DISS']
data_dict = make_data_dict(var_names,read_type)
PTB_HADV, PTB_VADV, PTB_HIDIFF, PTB_VIDIFF, PTB_HTURB, PTB_VTURB, PTB_MP, PTB_RAD, PTB_DIV, PTB_DISS = (data_dict[k] for k in var_names)

In [ ]:
################################################################################

In [ ]:
########################
#READING BACK IN
def LoadFinalData(in_file):
    dict = {}
    with h5py.File(in_file, 'r') as f:
        for key in f.keys():
            dict[key] = f[key][:]
    return dict

def LoadAllCloudBase():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"all_cloudbase_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        all_cloudbase = pickle.load(f)
    return(all_cloudbase)
min_all_cloudbase=np.nanmin(LoadAllCloudBase())
print(f"Minimum Cloudbase is: {min_all_cloudbase}\n")

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
in_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
final_dict=LoadFinalData(in_file)


#DYNAMICALLY CREATING VARIABLES
for key, value in final_dict.items():
    globals()[key] = value

# #DYNAMICALLY PRINTING VARIABLE SIZES
# for key in final_dict:
#     print(f"{key} has {final_dict[key].shape[0]} parcels")

# PRINTING VARIABLE SIZES (ONE BY ONE)
print(f'ALL: {len(CL_ALL_out_arr)} CL parcels and {len(nonCL_ALL_out_arr)} nonCL parcels')
print(f'SHALLOW: {len(CL_SHALLOW_out_arr)} CL parcels and {len(nonCL_SHALLOW_out_arr)} nonCL parcels')
print(f'DEEP: {len(CL_DEEP_out_arr)} CL parcels and {len(nonCL_DEEP_out_arr)} nonCL parcels')
print('\n')
print(f'ALL: {len(SBZ_ALL_out_arr)} SBZ parcels and {len(nonSBZ_ALL_out_arr)} nonSBZ parcels')
print(f'SHALLOW: {len(SBZ_SHALLOW_out_arr)} SBZ parcels and {len(nonSBZ_SHALLOW_out_arr)} nonSBZ parcels')
print(f'DEEP: {len(SBZ_DEEP_out_arr)} SBZ parcels and {len(nonSBZ_DEEP_out_arr)} nonSBZ parcels')
print('\n')
print(f'ALL: {len(ColdPool_ALL_out_arr)} ColdPool parcels')
print(f'SHALLOW: {len(ColdPool_SHALLOW_out_arr)} ColdPool parcels')
print(f'DEEP: {len(ColdPool_DEEP_out_arr)} ColdPool parcels')


#APPLYING JOB ARRAY
if "job_array" in globals():
    print('APPLYING JOB ARRAY')
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    for name in [
        'ALL_out_arr', 'ALL_save_arr',
        'SHALLOW_out_arr', 'SHALLOW_save_arr',
        'DEEP_out_arr', 'DEEP_save_arr',
        'ALL_SBZ_out_arr', 'ALL_nonSBZ_out_arr',
        'SHALLOW_SBZ_out_arr', 'SHALLOW_nonSBZ_out_arr',
        'DEEP_SBZ_out_arr', 'DEEP_nonSBZ_out_arr',
        'ALL_ColdPool_out_arr', 'SHALLOW_ColdPool_out_arr', 'DEEP_ColdPool_out_arr'
    ]:
        globals()[name] = job_filter(globals()[name])

In [ ]:
################################################################################


In [ ]:
#CL vs nonCL

In [ ]:
def CL_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_out_arr.copy()
        # after_array=ALL_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_out_arr.copy()
        # after_array=SHALLOW_out_after_array
    elif type=='deep':
        out_arr=DEEP_out_arr.copy()
        # after_array=DEEP_out_after_array

    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

In [ ]:
#Deep Parcels Lagrangian Parcels Profile (OPTIMIZED) 

#W Budgets
#'w': vertical velocity
#'w budget: horizontal advection (non-diff component)'
#'w budget: vertical advection (non-diff component)'
#'w budget: horiz implicit diffusion'
#'w budget: vert implicit diffusion'
#'w budget: horizontal parameterized turbulence'
#'w budget: vertical parameterized turbulence'
#'w budget: pressure gradient'
#'w budget: buoyancy'
variables = [W,WB_HADV,WB_VADV,WB_HIDIFF,WB_VIDIFF,WB_HTURB,WB_VTURB,WB_PGRAD,WB_BUOY]
types=['all','shallow','deep']
vars = [
    'w',
    'wb_hadv',
    'wb_vadv',
    'wb_hidiff',
    'wb_vidiff',
    'wb_hturb',
    'wb_vturb',
    'wb_pgrad',
    'wb_buoy'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=CL_tracked_profile(variable,type=type)

      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_all_WBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_shallow_WBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_deep_WBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_w', data=profile_w, compression="gzip")
        f.create_dataset('profile_wb_hadv', data=profile_wb_hadv, compression="gzip")
        f.create_dataset('profile_wb_vadv', data=profile_wb_vadv, compression="gzip")
        f.create_dataset('profile_wb_hidiff', data=profile_wb_hidiff, compression="gzip")
        f.create_dataset('profile_wb_vidiff', data=profile_wb_vidiff, compression="gzip")
        f.create_dataset('profile_wb_hturb', data=profile_wb_hturb, compression="gzip")
        f.create_dataset('profile_wb_vturb', data=profile_wb_vturb, compression="gzip")
        f.create_dataset('profile_wb_pgrad', data=profile_wb_pgrad, compression="gzip")
        f.create_dataset('profile_wb_buoy', data=profile_wb_buoy, compression="gzip")
    print('done')

In [ ]:
#CL QV Budgets

#'qv budget: horizontal advection (non-diff component)'
#'qv budget: vertical advection (non-diff component)'
#'qv budget: horiz implicit diffusion'
#'qv budget: vert implicit diffusion'
#'qv budget: horizontal parameterized turbulence'
#'qv budget: vertical parameterized turbulence'
#'qv budget: microphysics scheme'
variables = [QV,QVB_HADV,QVB_VADV,QVB_HIDIFF,QVB_VIDIFF,QVB_HTURB,QVB_VTURB,QVB_MP]


types=['all','shallow','deep']
vars = [
    'qv',
    'qvb_hadv',
    'qvb_vadv',
    'qvb_hidiff',
    'qvb_vidiff',
    'qvb_hturb',
    'qvb_vturb',
    'qvb_mp'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=CL_tracked_profile(variable,type=type)
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_all_QVBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_shallow_QVBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_deep_QVBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_qv', data=profile_qv, compression="gzip")
        f.create_dataset('profile_qvb_hadv', data=profile_qvb_hadv, compression="gzip")
        f.create_dataset('profile_qvb_vadv', data=profile_qvb_vadv, compression="gzip")
        f.create_dataset('profile_qvb_hidiff', data=profile_qvb_hidiff, compression="gzip")
        f.create_dataset('profile_qvb_vidiff', data=profile_qvb_vidiff, compression="gzip")
        f.create_dataset('profile_qvb_hturb', data=profile_qvb_hturb, compression="gzip")
        f.create_dataset('profile_qvb_vturb', data=profile_qvb_vturb, compression="gzip")
        f.create_dataset('profile_qvb_mp', data=profile_qvb_mp, compression="gzip")
    print('done')

In [ ]:
#CL TH Budgets

variables = [TH,PTB_HADV,PTB_VADV,PTB_HIDIFF,PTB_VIDIFF,PTB_HTURB,PTB_VTURB,PTB_MP]

types=['all','shallow','deep']
vars = [
    'th',
    'ptb_hadv',
    'ptb_vadv',
    'ptb_hidiff',
    'ptb_vidiff',
    'ptb_hturb',
    'ptb_vturb',
    'ptb_mp'
]

for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=CL_tracked_profile(variable,type=type)              
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_all_THBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_shallow_THBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/CL_deep_THBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_th', data=profile_th, compression="gzip")
        f.create_dataset('profile_ptb_hadv', data=profile_ptb_hadv, compression="gzip")
        f.create_dataset('profile_ptb_vadv', data=profile_ptb_vadv, compression="gzip")
        f.create_dataset('profile_ptb_hidiff', data=profile_ptb_hidiff, compression="gzip")
        f.create_dataset('profile_ptb_vidiff', data=profile_ptb_vidiff, compression="gzip")
        f.create_dataset('profile_ptb_hturb', data=profile_ptb_hturb, compression="gzip")
        f.create_dataset('profile_ptb_vturb', data=profile_ptb_vturb, compression="gzip")
        f.create_dataset('profile_ptb_mp', data=profile_ptb_mp, compression="gzip")
    print('done')

In [ ]:
def nonCL_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_save_arr.copy()
        # after_array=ALL_save_after_array
    elif type=='shallow':
        out_arr=SHALLOW_save_arr.copy()
        # after_array=SHALLOW_save_after_array
    elif type=='deep':
        out_arr=DEEP_save_arr.copy()
        # after_array=DEEP_save_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        vars=var_data[ts,p-index_adjust] #JOBARRAY
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

In [ ]:
#Deep Parcels Lagrangian Parcels Profile (OPTIMIZED) 

#nonCL W Budgets
#'w': vertical velocity
#'w budget: horizontal advection (non-diff component)'
#'w budget: vertical advection (non-diff component)'
#'w budget: horiz implicit diffusion'
#'w budget: vert implicit diffusion'
#'w budget: horizontal parameterized turbulence'
#'w budget: vertical parameterized turbulence'
#'w budget: pressure gradient'
#'w budget: buoyancy'
variables = [W,WB_HADV,WB_VADV,WB_HIDIFF,WB_VIDIFF,WB_HTURB,WB_VTURB,WB_PGRAD,WB_BUOY]
types=['all','shallow','deep']
vars = [
    'w',
    'wb_hadv',
    'wb_vadv',
    'wb_hidiff',
    'wb_vidiff',
    'wb_hturb',
    'wb_vturb',
    'wb_pgrad',
    'wb_buoy'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=nonCL_tracked_profile(variable,type=type)
              
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_all_WBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_shallow_WBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_deep_WBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_w', data=profile_w, compression="gzip")
        f.create_dataset('profile_wb_hadv', data=profile_wb_hadv, compression="gzip")
        f.create_dataset('profile_wb_vadv', data=profile_wb_vadv, compression="gzip")
        f.create_dataset('profile_wb_hidiff', data=profile_wb_hidiff, compression="gzip")
        f.create_dataset('profile_wb_vidiff', data=profile_wb_vidiff, compression="gzip")
        f.create_dataset('profile_wb_hturb', data=profile_wb_hturb, compression="gzip")
        f.create_dataset('profile_wb_vturb', data=profile_wb_vturb, compression="gzip")
        f.create_dataset('profile_wb_pgrad', data=profile_wb_pgrad, compression="gzip")
        f.create_dataset('profile_wb_buoy', data=profile_wb_buoy, compression="gzip")
    print('done')

In [ ]:
#nonCL QV Budgets

#'qv budget: horizontal advection (non-diff component)'
#'qv budget: vertical advection (non-diff component)'
#'qv budget: horiz implicit diffusion'
#'qv budget: vert implicit diffusion'
#'qv budget: horizontal parameterized turbulence'
#'qv budget: vertical parameterized turbulence'
#'qv budget: microphysics scheme'
variables = [QV,QVB_HADV,QVB_VADV,QVB_HIDIFF,QVB_VIDIFF,QVB_HTURB,QVB_VTURB,QVB_MP]


types=['all','shallow','deep']
vars = [
    'qv',
    'qvb_hadv',
    'qvb_vadv',
    'qvb_hidiff',
    'qvb_vidiff',
    'qvb_hturb',
    'qvb_vturb',
    'qvb_mp'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=nonCL_tracked_profile(variable,type=type)
              
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_all_QVBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_shallow_QVBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_deep_QVBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_qv', data=profile_qv, compression="gzip")
        f.create_dataset('profile_qvb_hadv', data=profile_qvb_hadv, compression="gzip")
        f.create_dataset('profile_qvb_vadv', data=profile_qvb_vadv, compression="gzip")
        f.create_dataset('profile_qvb_hidiff', data=profile_qvb_hidiff, compression="gzip")
        f.create_dataset('profile_qvb_vidiff', data=profile_qvb_vidiff, compression="gzip")
        f.create_dataset('profile_qvb_hturb', data=profile_qvb_hturb, compression="gzip")
        f.create_dataset('profile_qvb_vturb', data=profile_qvb_vturb, compression="gzip")
        f.create_dataset('profile_qvb_mp', data=profile_qvb_mp, compression="gzip")
    print('done')

In [ ]:
#CL TH Budgets

variables = [TH,PTB_HADV,PTB_VADV,PTB_HIDIFF,PTB_VIDIFF,PTB_HTURB,PTB_VTURB,PTB_MP]

types=['all','shallow','deep']
vars = [
    'th',
    'ptb_hadv',
    'ptb_vadv',
    'ptb_hidiff',
    'ptb_vidiff',
    'ptb_hturb',
    'ptb_vturb',
    'ptb_mp'
]

for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=nonCL_tracked_profile(variable,type=type) 
              
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_all_THBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_shallow_THBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/nonCL_deep_THBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_th', data=profile_th, compression="gzip")
        f.create_dataset('profile_ptb_hadv', data=profile_ptb_hadv, compression="gzip")
        f.create_dataset('profile_ptb_vadv', data=profile_ptb_vadv, compression="gzip")
        f.create_dataset('profile_ptb_hidiff', data=profile_ptb_hidiff, compression="gzip")
        f.create_dataset('profile_ptb_vidiff', data=profile_ptb_vidiff, compression="gzip")
        f.create_dataset('profile_ptb_hturb', data=profile_ptb_hturb, compression="gzip")
        f.create_dataset('profile_ptb_vturb', data=profile_ptb_vturb, compression="gzip")
        f.create_dataset('profile_ptb_mp', data=profile_ptb_mp, compression="gzip")
    print('done')

In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER

recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
if recombine==True:
    def read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id):
    
    
        if BUDGET_var=='W':
            vars_list=['profile_w', 'profile_wb_hadv', 'profile_wb_vadv', 'profile_wb_hidiff', 
             'profile_wb_vidiff', 'profile_wb_hturb', 'profile_wb_vturb', 'profile_wb_pgrad', 
             'profile_wb_buoy']
        elif BUDGET_var=='QV':
            vars_list=['profile_qv', 'profile_qvb_hadv', 'profile_qvb_vadv', 
             'profile_qvb_hidiff', 'profile_qvb_vidiff', 'profile_qvb_hturb', 'profile_qvb_vturb', 
             'profile_qvb_mp']
        elif BUDGET_var=='TH':
            vars_list=['profile_th', 'profile_ptb_hadv', 'profile_ptb_vadv', 
             'profile_ptb_hidiff', 'profile_ptb_vidiff', 'profile_ptb_hturb', 'profile_ptb_vturb', 
             'profile_ptb_mp']
    
        input_file=dir+f"Project_Algorithms/plots/job_out/{CLnonCL}_{allshallowdeep}_{BUDGET_var}BUDGET_profile_{allshallowdeep}_{job_id}"
        input_file+='_5min.h5'
        with h5py.File(input_file, 'r') as f:
            dict = {}
            for var in vars_list:
                dict[var] = f[var][:]
        return dict

In [ ]:
if recombine==True: #*#*#*#
    CLnonCLs=['CL','nonCL']
    allshallowdeeps=['all','shallow','deep']
    BUDGET_vars=['W','QV','TH']
    
    for CLnonCL in CLnonCLs:
        print('\n'+CLnonCL)
        for allshallowdeep in allshallowdeeps:
            print(allshallowdeep)
            for BUDGET_var in BUDGET_vars:
                job_id=1
                dict1=read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id)
                
                num_jobs=60
                for job_id in np.arange(2,num_jobs+1):
                    
                    dict2=read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id)
                    for var in dict2:
                        dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
                
                output_file=dir+f"Project_Algorithms/plots/job_out/{CLnonCL}_{allshallowdeep}_{BUDGET_var}BUDGET_profile_{allshallowdeep}"
                output_file+='_5min.h5'
                with h5py.File(output_file, 'w') as f:
                    for var in dict1:
                        profile_var = dict1[var]
                        f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [ ]:
#################################
#SBZ and nonSBZ

In [34]:
#SBZ
def SBZ_tracked_profile(var_data,type):

    if type=='all':
        out_arr=ALL_SBZ_out_arr.copy()
        # after_array=ALL_SBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_SBZ_out_arr.copy()
        # after_array=SHALLOW_SBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_SBZ_out_arr.copy()
        # after_array=DEEP_SBZ_out_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

In [35]:
#Deep Parcels Lagrangian Parcels Profile (OPTIMIZED) 

#W Budgets
#'w': vertical velocity
#'w budget: horizontal advection (non-diff component)'
#'w budget: vertical advection (non-diff component)'
#'w budget: horiz implicit diffusion'
#'w budget: vert implicit diffusion'
#'w budget: horizontal parameterized turbulence'
#'w budget: vertical parameterized turbulence'
#'w budget: pressure gradient'
#'w budget: buoyancy'
variables = [W,WB_HADV,WB_VADV,WB_HIDIFF,WB_VIDIFF,WB_HTURB,WB_VTURB,WB_PGRAD,WB_BUOY]
types=['all','shallow','deep']
vars = [
    'w',
    'wb_hadv',
    'wb_vadv',
    'wb_hidiff',
    'wb_vidiff',
    'wb_hturb',
    'wb_vturb',
    'wb_pgrad',
    'wb_buoy'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=SBZ_tracked_profile(variable,type=type)
              
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_all_WBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_shallow_WBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_deep_WBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_w', data=profile_w, compression="gzip")
        f.create_dataset('profile_wb_hadv', data=profile_wb_hadv, compression="gzip")
        f.create_dataset('profile_wb_vadv', data=profile_wb_vadv, compression="gzip")
        f.create_dataset('profile_wb_hidiff', data=profile_wb_hidiff, compression="gzip")
        f.create_dataset('profile_wb_vidiff', data=profile_wb_vidiff, compression="gzip")
        f.create_dataset('profile_wb_hturb', data=profile_wb_hturb, compression="gzip")
        f.create_dataset('profile_wb_vturb', data=profile_wb_vturb, compression="gzip")
        f.create_dataset('profile_wb_pgrad', data=profile_wb_pgrad, compression="gzip")
        f.create_dataset('profile_wb_buoy', data=profile_wb_buoy, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [36]:
#CL QV Budgets

#'qv budget: horizontal advection (non-diff component)'
#'qv budget: vertical advection (non-diff component)'
#'qv budget: horiz implicit diffusion'
#'qv budget: vert implicit diffusion'
#'qv budget: horizontal parameterized turbulence'
#'qv budget: vertical parameterized turbulence'
#'qv budget: microphysics scheme'
variables = [QV,QVB_HADV,QVB_VADV,QVB_HIDIFF,QVB_VIDIFF,QVB_HTURB,QVB_VTURB,QVB_MP]


types=['all','shallow','deep']
vars = [
    'qv',
    'qvb_hadv',
    'qvb_vadv',
    'qvb_hidiff',
    'qvb_vidiff',
    'qvb_hturb',
    'qvb_vturb',
    'qvb_mp'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=SBZ_tracked_profile(variable,type=type)
              
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_all_QVBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_shallow_QVBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_deep_QVBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_qv', data=profile_qv, compression="gzip")
        f.create_dataset('profile_qvb_hadv', data=profile_qvb_hadv, compression="gzip")
        f.create_dataset('profile_qvb_vadv', data=profile_qvb_vadv, compression="gzip")
        f.create_dataset('profile_qvb_hidiff', data=profile_qvb_hidiff, compression="gzip")
        f.create_dataset('profile_qvb_vidiff', data=profile_qvb_vidiff, compression="gzip")
        f.create_dataset('profile_qvb_hturb', data=profile_qvb_hturb, compression="gzip")
        f.create_dataset('profile_qvb_vturb', data=profile_qvb_vturb, compression="gzip")
        f.create_dataset('profile_qvb_mp', data=profile_qvb_mp, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [37]:
#CL TH Budgets

variables = [TH,PTB_HADV,PTB_VADV,PTB_HIDIFF,PTB_VIDIFF,PTB_HTURB,PTB_VTURB,PTB_MP]

types=['all','shallow','deep']
vars = [
    'th',
    'ptb_hadv',
    'ptb_vadv',
    'ptb_hidiff',
    'ptb_vidiff',
    'ptb_hturb',
    'ptb_vturb',
    'ptb_mp'
]

for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=SBZ_tracked_profile(variable,type=type) 
              
            
      
    if type=='all':   
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_all_THBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_shallow_THBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/SBZ_deep_THBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_th', data=profile_th, compression="gzip")
        f.create_dataset('profile_ptb_hadv', data=profile_ptb_hadv, compression="gzip")
        f.create_dataset('profile_ptb_vadv', data=profile_ptb_vadv, compression="gzip")
        f.create_dataset('profile_ptb_hidiff', data=profile_ptb_hidiff, compression="gzip")
        f.create_dataset('profile_ptb_vidiff', data=profile_ptb_vidiff, compression="gzip")
        f.create_dataset('profile_ptb_hturb', data=profile_ptb_hturb, compression="gzip")
        f.create_dataset('profile_ptb_vturb', data=profile_ptb_vturb, compression="gzip")
        f.create_dataset('profile_ptb_mp', data=profile_ptb_mp, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [38]:
def nonSBZ_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_nonSBZ_out_arr.copy()
        # after_array=ALL_nonSBZ_out_after_array.copy()
    elif type=='shallow':
        out_arr=SHALLOW_nonSBZ_out_arr.copy()
        # after_array=SHALLOW_nonSBZ_out_after_array.copy()
    elif type=='deep':
        out_arr=DEEP_nonSBZ_out_arr.copy()
        # after_array=DEEP_nonSBZ_out_after_array.copy()
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p]
        ys=Y[ts,p]
        xs=X[ts,p]
        vars=var_data[ts,p]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

In [39]:
#Deep Parcels Lagrangian Parcels Profile (OPTIMIZED) 

#W Budgets
#'w': vertical velocity
#'w budget: horizontal advection (non-diff component)'
#'w budget: vertical advection (non-diff component)'
#'w budget: horiz implicit diffusion'
#'w budget: vert implicit diffusion'
#'w budget: horizontal parameterized turbulence'
#'w budget: vertical parameterized turbulence'
#'w budget: pressure gradient'
#'w budget: buoyancy'
variables = [W,WB_HADV,WB_VADV,WB_HIDIFF,WB_VIDIFF,WB_HTURB,WB_VTURB,WB_PGRAD,WB_BUOY]
types=['all','shallow','deep']
vars = [
    'w',
    'wb_hadv',
    'wb_vadv',
    'wb_hidiff',
    'wb_vidiff',
    'wb_hturb',
    'wb_vturb',
    'wb_pgrad',
    'wb_buoy'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=nonSBZ_tracked_profile(variable,type=type)
              
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_all_WBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_shallow_WBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_deep_WBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_w', data=profile_w, compression="gzip")
        f.create_dataset('profile_wb_hadv', data=profile_wb_hadv, compression="gzip")
        f.create_dataset('profile_wb_vadv', data=profile_wb_vadv, compression="gzip")
        f.create_dataset('profile_wb_hidiff', data=profile_wb_hidiff, compression="gzip")
        f.create_dataset('profile_wb_vidiff', data=profile_wb_vidiff, compression="gzip")
        f.create_dataset('profile_wb_hturb', data=profile_wb_hturb, compression="gzip")
        f.create_dataset('profile_wb_vturb', data=profile_wb_vturb, compression="gzip")
        f.create_dataset('profile_wb_pgrad', data=profile_wb_pgrad, compression="gzip")
        f.create_dataset('profile_wb_buoy', data=profile_wb_buoy, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [40]:
#CL QV Budgets

#'qv budget: horizontal advection (non-diff component)'
#'qv budget: vertical advection (non-diff component)'
#'qv budget: horiz implicit diffusion'
#'qv budget: vert implicit diffusion'
#'qv budget: horizontal parameterized turbulence'
#'qv budget: vertical parameterized turbulence'
#'qv budget: microphysics scheme'
variables = [QV,QVB_HADV,QVB_VADV,QVB_HIDIFF,QVB_VIDIFF,QVB_HTURB,QVB_VTURB,QVB_MP]


types=['all','shallow','deep']
vars = [
    'qv',
    'qvb_hadv',
    'qvb_vadv',
    'qvb_hidiff',
    'qvb_vidiff',
    'qvb_hturb',
    'qvb_vturb',
    'qvb_mp'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=nonSBZ_tracked_profile(variable,type=type)
              
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_all_QVBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_shallow_QVBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_deep_QVBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_qv', data=profile_qv, compression="gzip")
        f.create_dataset('profile_qvb_hadv', data=profile_qvb_hadv, compression="gzip")
        f.create_dataset('profile_qvb_vadv', data=profile_qvb_vadv, compression="gzip")
        f.create_dataset('profile_qvb_hidiff', data=profile_qvb_hidiff, compression="gzip")
        f.create_dataset('profile_qvb_vidiff', data=profile_qvb_vidiff, compression="gzip")
        f.create_dataset('profile_qvb_hturb', data=profile_qvb_hturb, compression="gzip")
        f.create_dataset('profile_qvb_vturb', data=profile_qvb_vturb, compression="gzip")
        f.create_dataset('profile_qvb_mp', data=profile_qvb_mp, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [41]:
#CL TH Budgets

variables = [TH,PTB_HADV,PTB_VADV,PTB_HIDIFF,PTB_VIDIFF,PTB_HTURB,PTB_VTURB,PTB_MP]

types=['all','shallow','deep']
vars = [
    'th',
    'ptb_hadv',
    'ptb_vadv',
    'ptb_hidiff',
    'ptb_vidiff',
    'ptb_hturb',
    'ptb_vturb',
    'ptb_mp'
]

for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=nonSBZ_tracked_profile(variable,type=type)    
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_all_THBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_shallow_THBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/nonSBZ_deep_THBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_th', data=profile_th, compression="gzip")
        f.create_dataset('profile_ptb_hadv', data=profile_ptb_hadv, compression="gzip")
        f.create_dataset('profile_ptb_vadv', data=profile_ptb_vadv, compression="gzip")
        f.create_dataset('profile_ptb_hidiff', data=profile_ptb_hidiff, compression="gzip")
        f.create_dataset('profile_ptb_vidiff', data=profile_ptb_vidiff, compression="gzip")
        f.create_dataset('profile_ptb_hturb', data=profile_ptb_hturb, compression="gzip")
        f.create_dataset('profile_ptb_vturb', data=profile_ptb_vturb, compression="gzip")
        f.create_dataset('profile_ptb_mp', data=profile_ptb_mp, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [42]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER

recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [43]:
if recombine==True:
    def read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id):
    
    
        if BUDGET_var=='W':
            vars_list=['profile_w', 'profile_wb_hadv', 'profile_wb_vadv', 'profile_wb_hidiff', 
             'profile_wb_vidiff', 'profile_wb_hturb', 'profile_wb_vturb', 'profile_wb_pgrad', 
             'profile_wb_buoy']
        elif BUDGET_var=='QV':
            vars_list=['profile_qv', 'profile_qvb_hadv', 'profile_qvb_vadv', 
             'profile_qvb_hidiff', 'profile_qvb_vidiff', 'profile_qvb_hturb', 'profile_qvb_vturb', 
             'profile_qvb_mp']
        elif BUDGET_var=='TH':
            vars_list=['profile_th', 'profile_ptb_hadv', 'profile_ptb_vadv', 
             'profile_ptb_hidiff', 'profile_ptb_vidiff', 'profile_ptb_hturb', 'profile_ptb_vturb', 
             'profile_ptb_mp']
    
        input_file=dir+f"Project_Algorithms/plots/job_out/{CLnonCL}_{allshallowdeep}_{BUDGET_var}BUDGET_profile_{allshallowdeep}_{job_id}"
        input_file+='_5min.h5'
        with h5py.File(input_file, 'r') as f:
            dict = {}
            for var in vars_list:
                dict[var] = f[var][:]
        return dict

In [44]:
if recombine==True: #*#*#*#
    CLnonCLs=['SBZ','nonSBZ']
    allshallowdeeps=['all','shallow','deep']
    BUDGET_vars=['W','QV','TH']
    
    for CLnonCL in CLnonCLs:
        print('\n'+CLnonCL)
        for allshallowdeep in allshallowdeeps:
            print(allshallowdeep)
            for BUDGET_var in BUDGET_vars:
                job_id=1
                dict1=read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id)
                
                num_jobs=60
                for job_id in np.arange(2,num_jobs+1):
                    
                    dict2=read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id)
                    for var in dict2:
                        dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
                
                output_file=dir+f"Project_Algorithms/plots/job_out/{CLnonCL}_{allshallowdeep}_{BUDGET_var}BUDGET_profile_{allshallowdeep}"
                output_file+='_5min.h5'
                with h5py.File(output_file, 'w') as f:
                    for var in dict1:
                        profile_var = dict1[var]
                        f.create_dataset(f'{var}', data=profile_var, compression="gzip")


SBZ
all
shallow
deep

nonSBZ
all
shallow
deep


In [ ]:
#ColdPool
################################################################

In [52]:
def ColdPool_tracked_profile(var_data,type):

    if type=='all':
        out_arr=ALL_ColdPool_out_arr.copy()
        # after_array=ALL_ColdPool_after_array
    elif type=='shallow':
        out_arr=SHALLOW_ColdPool_out_arr
        # after_array=SHALLOW_ColdPool_after_array
    elif type=='deep':
        out_arr=DEEP_ColdPool_out_arr
        # after_array=DEEP_ColdPool_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p]
        ys=Y[ts,p]
        xs=X[ts,p]
        vars=var_data[ts,p]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)

    return profile_array

In [53]:
#Deep Parcels Lagrangian Parcels Profile (OPTIMIZED) 

#W Budgets
#'w': vertical velocity
#'w budget: horizontal advection (non-diff component)'
#'w budget: vertical advection (non-diff component)'
#'w budget: horiz implicit diffusion'
#'w budget: vert implicit diffusion'
#'w budget: horizontal parameterized turbulence'
#'w budget: vertical parameterized turbulence'
#'w budget: pressure gradient'
#'w budget: buoyancy'
variables = [W,WB_HADV,WB_VADV,WB_HIDIFF,WB_VIDIFF,WB_HTURB,WB_VTURB,WB_PGRAD,WB_BUOY]
types=['all','shallow','deep']
vars = [
    'w',
    'wb_hadv',
    'wb_vadv',
    'wb_hidiff',
    'wb_vidiff',
    'wb_hturb',
    'wb_vturb',
    'wb_pgrad',
    'wb_buoy'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=ColdPool_tracked_profile(variable,type=type)
              
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_all_WBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_shallow_WBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_deep_WBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_w', data=profile_w, compression="gzip")
        f.create_dataset('profile_wb_hadv', data=profile_wb_hadv, compression="gzip")
        f.create_dataset('profile_wb_vadv', data=profile_wb_vadv, compression="gzip")
        f.create_dataset('profile_wb_hidiff', data=profile_wb_hidiff, compression="gzip")
        f.create_dataset('profile_wb_vidiff', data=profile_wb_vidiff, compression="gzip")
        f.create_dataset('profile_wb_hturb', data=profile_wb_hturb, compression="gzip")
        f.create_dataset('profile_wb_vturb', data=profile_wb_vturb, compression="gzip")
        f.create_dataset('profile_wb_pgrad', data=profile_wb_pgrad, compression="gzip")
        f.create_dataset('profile_wb_buoy', data=profile_wb_buoy, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [54]:
#ColdPool QV Budgets

#'qv budget: horizontal advection (non-diff component)'
#'qv budget: vertical advection (non-diff component)'
#'qv budget: horiz implicit diffusion'
#'qv budget: vert implicit diffusion'
#'qv budget: horizontal parameterized turbulence'
#'qv budget: vertical parameterized turbulence'
#'qv budget: microphysics scheme'
variables = [QV,QVB_HADV,QVB_VADV,QVB_HIDIFF,QVB_VIDIFF,QVB_HTURB,QVB_VTURB,QVB_MP]


types=['all','shallow','deep']
vars = [
    'qv',
    'qvb_hadv',
    'qvb_vadv',
    'qvb_hidiff',
    'qvb_vidiff',
    'qvb_hturb',
    'qvb_vturb',
    'qvb_mp'
]
for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=ColdPool_tracked_profile(variable,type=type)
              
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_all_QVBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_shallow_QVBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_deep_QVBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_qv', data=profile_qv, compression="gzip")
        f.create_dataset('profile_qvb_hadv', data=profile_qvb_hadv, compression="gzip")
        f.create_dataset('profile_qvb_vadv', data=profile_qvb_vadv, compression="gzip")
        f.create_dataset('profile_qvb_hidiff', data=profile_qvb_hidiff, compression="gzip")
        f.create_dataset('profile_qvb_vidiff', data=profile_qvb_vidiff, compression="gzip")
        f.create_dataset('profile_qvb_hturb', data=profile_qvb_hturb, compression="gzip")
        f.create_dataset('profile_qvb_vturb', data=profile_qvb_vturb, compression="gzip")
        f.create_dataset('profile_qvb_mp', data=profile_qvb_mp, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [55]:
#ColdPool TH Budgets

variables = [TH,PTB_HADV,PTB_VADV,PTB_HIDIFF,PTB_VIDIFF,PTB_HTURB,PTB_VTURB,PTB_MP]

types=['all','shallow','deep']
vars = [
    'th',
    'ptb_hadv',
    'ptb_vadv',
    'ptb_hidiff',
    'ptb_vidiff',
    'ptb_hturb',
    'ptb_vturb',
    'ptb_mp'
]

for type in types:
    print(f"type {type}")

    # #creates profile storage and adds z column  
    for var in vars:
        zhs=data['zh'].values
        globals()[f"profile_{var}"]=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
        globals()[f"profile_{var}"][:,2]=zhs

    
    for (var,variable) in zip(vars,variables):
            globals()[f"profile_{var}"]=ColdPool_tracked_profile(variable,type=type) 
              
            
      
    if type=='all':    
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_all_THBUDGET_profile_all_{job_id}'
        output_file+='_5min.h5'
    elif type=='shallow': 
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_shallow_THBUDGET_profile_shallow_{job_id}'
        output_file+='_5min.h5'
    elif type=='deep':  
        output_file=dir+f'Project_Algorithms/plots/job_out/ColdPool_deep_THBUDGET_profile_deep_{job_id}'
        output_file+='_5min.h5'
    import h5py
    with h5py.File(output_file, 'w') as f:
        f.create_dataset('profile_th', data=profile_th, compression="gzip")
        f.create_dataset('profile_ptb_hadv', data=profile_ptb_hadv, compression="gzip")
        f.create_dataset('profile_ptb_vadv', data=profile_ptb_vadv, compression="gzip")
        f.create_dataset('profile_ptb_hidiff', data=profile_ptb_hidiff, compression="gzip")
        f.create_dataset('profile_ptb_vidiff', data=profile_ptb_vidiff, compression="gzip")
        f.create_dataset('profile_ptb_hturb', data=profile_ptb_hturb, compression="gzip")
        f.create_dataset('profile_ptb_vturb', data=profile_ptb_vturb, compression="gzip")
        f.create_dataset('profile_ptb_mp', data=profile_ptb_mp, compression="gzip")
    print('done')

type all
done
type shallow
done
type deep
done


In [56]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER

recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [57]:
if recombine==True:
    def read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id):
    
    
        if BUDGET_var=='W':
            vars_list=['profile_w', 'profile_wb_hadv', 'profile_wb_vadv', 'profile_wb_hidiff', 
             'profile_wb_vidiff', 'profile_wb_hturb', 'profile_wb_vturb', 'profile_wb_pgrad', 
             'profile_wb_buoy']
        elif BUDGET_var=='QV':
            vars_list=['profile_qv', 'profile_qvb_hadv', 'profile_qvb_vadv', 
             'profile_qvb_hidiff', 'profile_qvb_vidiff', 'profile_qvb_hturb', 'profile_qvb_vturb', 
             'profile_qvb_mp']
        elif BUDGET_var=='TH':
            vars_list=['profile_th', 'profile_ptb_hadv', 'profile_ptb_vadv', 
             'profile_ptb_hidiff', 'profile_ptb_vidiff', 'profile_ptb_hturb', 'profile_ptb_vturb', 
             'profile_ptb_mp']
    
        input_file=dir+f"Project_Algorithms/plots/job_out/{CLnonCL}_{allshallowdeep}_{BUDGET_var}BUDGET_profile_{allshallowdeep}_{job_id}"
        input_file+='_5min.h5'
        with h5py.File(input_file, 'r') as f:
            dict = {}
            for var in vars_list:
                dict[var] = f[var][:]
        return dict

In [58]:
if recombine==True: #*#*#*#
    CLnonCLs=['ColdPool']
    allshallowdeeps=['all','shallow','deep']
    BUDGET_vars=['W','QV','TH']
    
    for CLnonCL in CLnonCLs:
        print('\n'+CLnonCL)
        for allshallowdeep in allshallowdeeps:
            print(allshallowdeep)
            for BUDGET_var in BUDGET_vars:
                job_id=1
                dict1=read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id)
                
                num_jobs=60
                for job_id in np.arange(2,num_jobs+1):
                    
                    dict2=read_budget_profiles(CLnonCL,allshallowdeep,BUDGET_var,job_id)
                    for var in dict2:
                        dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
                
                output_file=dir+f"Project_Algorithms/plots/job_out/{CLnonCL}_{allshallowdeep}_{BUDGET_var}BUDGET_profile_{allshallowdeep}"
                output_file+='_5min.h5'
                with h5py.File(output_file, 'w') as f:
                    for var in dict1:
                        profile_var = dict1[var]
                        f.create_dataset(f'{var}', data=profile_var, compression="gzip")


ColdPool
all
shallow
deep
